# Molecular Standardization

## Scientific objective
Apply a transparent RDKit structure pipeline while retaining original structures, standardized structures, InChI/InChIKey, Bemis–Murcko scaffolds, statuses, failures, and transformation logs.

## Inputs
- Raw tables from notebook 02
- Standardization configuration

## Expected outputs
- `data/interim/standardized_long.csv` (and Parquet when available)
- `data/metadata/standardization_summary.json`

## Dependencies
RDKit, pandas

## Reproducibility seed
`20260723`. The seed is loaded from `configs/training_config.yaml`; split files and checkpoints are persisted.

## Data and model assumptions
The principal fragment is the largest organic fragment by heavy atoms, then molecular weight. Stereochemistry is preserved; isotopes are retained by default.

## Validation checks
The executable cells below fail explicitly on missing/inconsistent required artifacts and save machine-readable status records.

## Interpretation of results
Interpret endpoint-level outputs only after checking prevalence, missingness, split integrity, calibration, uncertainty, and applicability-domain coverage. No notebook result is evidence that experimental toxicity testing can be replaced.

## Saved artifacts
Artifacts listed above are written under `data/`, `models/`, `results/`, `figures/`, `tables/`, or `reports/` and are consumed by later notebooks.

## Limitations
Fragment selection and uncharging can alter the assay-relevant entity. Every change is logged for audit and sensitivity analysis.

## Next notebook
[05_data_quality_and_duplicate_audit.ipynb](./05_data_quality_and_duplicate_audit.ipynb)

In [1]:
from pathlib import Path
import os, json, warnings
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the repository root or notebooks directory")
os.chdir(ROOT)

from toxicity_screening.config import load_configs, execution_profile
from toxicity_screening.utils import set_global_seed, require_paths

CONFIGS = load_configs(ROOT)
PROFILE, PROFILE_CONFIG = execution_profile(CONFIGS)
SEED = int(CONFIGS["training_config"]["seed"])
set_global_seed(SEED)
print({"root": str(ROOT), "profile": PROFILE, "seed": SEED})

{'root': 'D:\\Dropbox\\Work\\Learning\\Python\\toxicity_screening_project', 'profile': 'smoke', 'seed': 20260723}


In [2]:
from toxicity_screening.pipeline import standardize_all

derived = standardize_all(ROOT)

display(
    derived[
        [
            "original_smiles",
            "standardized_smiles",
            "inchikey",
            "scaffold",
            "standardization_status",
            "failure_reason",
        ]
    ].head()
)

,original_smiles,standardized_smiles,inchikey,scaffold,standardization_status,failure_reason
0,Fc1ccc(-n2cc(NCCN3CCCCC3)nn2)cc1F,Fc1ccc(-n2cc(NCCN3CCCCC3)nn2)cc1F,PMWZSLRPVRCNKM-UHFFFAOYSA-N,c1ccc(-n2cc(NCCN3CCCCC3)nn2)cc1,success,None
1,COc1cc(N2Cc3ccc(Sc4ccc(F)cc4)nc3C2=O)ccc1OCCN1...,COc1cc(N2Cc3ccc(Sc4ccc(F)cc4)nc3C2=O)ccc1OCCN1...,IOVNPTLMMFSPCO-UHFFFAOYSA-N,O=C1c2nc(Sc3ccccc3)ccc2CN1c1ccc(OCCN2CCCC2)cc1,success,None
2,CCOC(=O)[C@H]1CC[C@@H](N2CC(NC(=O)CNc3nn(C(N)=...,CCOC(=O)[C@H]1CC[C@@H](N2CC(NC(=O)CNc3nn(C(N)=...,BSPCGOZVENCINN-AKAXFMLLSA-N,O=C(CNc1n[nH]c2ccccc12)NC1CN(C2CCCCC2)C1,success,None
3,N[C@@H](Cn1c(=O)cnc2ccc(F)cc21)C1CCC(NCc2ccc3c...,N[C@@H](Cn1c(=O)cnc2ccc(F)cc21)C1CCC(NCc2ccc3c...,UQBOMAVTFFNGLT-PVARCSIZSA-N,O=C1COc2ccc(CNC3CCC(CCn4c(=O)cnc5ccccc54)CC3)n...,success,None
4,O=C(NC1COc2cccc(-c3ccnc(CO)c3)c2C1)c1ccc(OCC(F...,O=C(NC1COc2cccc(-c3ccnc(CO)c3)c2C1)c1ccc(OCC(F...,DEYZGKXTVSBRIY-UHFFFAOYSA-N,O=C(NC1COc2cccc(-c3ccncc3)c2C1)c1cccnc1,success,None


In [4]:
assert len(derived) > 0
assert derived.loc[derived["standardization_status"] == "success", "standardized_smiles"].notna().all()
failed = derived.loc[derived["standardization_status"] != "success", ["original_smiles", "failure_reason"]]
failed.to_csv(ROOT / "data/metadata/standardization_failures.csv", index=False)
print({"records": len(derived), "failed": len(failed), "failure_rate": len(failed)/len(derived)})

{'records': 52047, 'failed': 4, 'failure_rate': 7.685361308048494e-05}


**Inspect the failed structures**

In [5]:
display(
    failed[
        ["original_smiles", "failure_reason"]
    ].drop_duplicates()
)

,original_smiles,failure_reason
27270,[SbH6+3],AtomValenceException: Explicit valence for ato...


In [6]:
failed["failure_reason"].value_counts(dropna=False)

failure_reason
AtomValenceException: Explicit valence for atom # 0 Sb, 9, is greater than permitted    4
Name: count, dtype: int64

### Completion gate
Confirm that the declared artifacts exist before continuing to `05_data_quality_and_duplicate_audit.ipynb`.